# 🧠 Poker Arena — Notebook 03: treinar a IA (PPO self-play)

A rede neural vai jogar **milhões de mãos contra si mesma** e aprender padrões, value bet e **blefe** sozinha. Quanto mais treino, mais forte — e **cada checkpoint salvo vira um nível de dificuldade** no jogo (Master = meio do treino, Expert = final).

**Roda no Colab Pro+** (GPU). **Segredos necessários** (🔑 Secrets do Colab): `HF_TOKEN` (publicar o modelo) e `GH_TOKEN` (um PAT do GitHub, pra instalar o repo privado).

> Treino grande = horas. Use **GPU** + **execução em background** (Pro+). Aumente `TOTAL_TIMESTEPS` pra treinar mais.

In [ ]:
# 1) Dependencias + auth + instala o pacote poker_arena (repo privado via token)
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
gh = userdata.get('GH_TOKEN')  # PAT do GitHub com acesso ao repo privado
assert gh, 'Adicione um segredo GH_TOKEN (PAT do GitHub) nos Secrets do Colab.'

!pip install -q stable-baselines3 sb3-contrib gymnasium onnx huggingface_hub
!pip install -q 'git+https://{gh}@github.com/FELIPEACASTRO/poker-arena.git#subdirectory=backend'
print('Instalado. (NAO compartilhe a saida desta celula - contem o token)')

In [ ]:
# 2) Imports + o motor embrulhado como env Gym (com mascara de acoes legais)
import numpy as np
import torch
import gymnasium as gym
from gymnasium import spaces
from sb3_contrib import MaskablePPO
from stable_baselines3.common.callbacks import BaseCallback
from poker_arena.ml import FEATURE_SIZE, N_ACTIONS, SelfPlayEnv

def box(titulo, linhas):
    w = max([len(titulo)] + [len(l) for l in linhas])
    print('  +' + '-'*(w+2) + '+'); print('  | ' + titulo.ljust(w) + ' |'); print('  +' + '-'*(w+2) + '+')
    for l in linhas: print('  | ' + l.ljust(w) + ' |')
    print('  +' + '-'*(w+2) + '+')

class GymPokerEnv(gym.Env):
    def __init__(self, n_players=6):
        super().__init__()
        self.env = SelfPlayEnv(n_players=n_players)
        self.observation_space = spaces.Box(0.0, 5.0, (FEATURE_SIZE,), dtype=np.float32)
        self.action_space = spaces.Discrete(N_ACTIONS)
        self._mask = [True]*N_ACTIONS
    def reset(self, *, seed=None, options=None):
        obs, self._mask = self.env.reset()
        return np.asarray(obs, dtype=np.float32), {}
    def step(self, action):
        obs, self._mask, reward, done = self.env.step(int(action))
        return np.asarray(obs, dtype=np.float32), float(reward), bool(done), False, {}
    def action_masks(self):
        return np.asarray(self._mask, dtype=bool)

box('NOTEBOOK 03 - TREINANDO A IA (PPO self-play)', [
    'A rede joga milhoes de maos contra si mesma e aprende padroes,',
    'value bet e BLEFE sozinha. Mais treino = mais forte.',
    'Cada checkpoint salvo vira um nivel de dificuldade no jogo.',
])

In [ ]:
# 3) Monta o modelo + liga o self-play (o oponente vira a propria politica)
N_PLAYERS = 6
TOTAL_TIMESTEPS = 2_000_000   # <-- AUMENTE para treinar mais (Pro+ A100 aguenta)
CKPT_EVERY = 500_000          # salva um nivel a cada tantos passos

gym_env = GymPokerEnv(n_players=N_PLAYERS)
model = MaskablePPO('MlpPolicy', gym_env, n_steps=2048, batch_size=256,
                    gamma=0.99, gae_lambda=0.95, ent_coef=0.01, verbose=0,
                    policy_kwargs=dict(net_arch=[256, 256]), device='auto')

def model_opponent(obs, mask):
    a, _ = model.predict(np.asarray(obs, dtype=np.float32),
                         action_masks=np.asarray(mask, dtype=bool), deterministic=False)
    return int(a)
gym_env.env.set_opponent(model_opponent)

box('CONFIG DO TREINO', [
    f'jogadores na mesa : {N_PLAYERS}',
    f'passos de treino  : {TOTAL_TIMESTEPS:,}',
    f'checkpoint a cada : {CKPT_EVERY:,} passos (= 1 nivel)',
    f'device            : {model.device}',
])

In [ ]:
# 4) Treina (acompanhe o lucro medio por mao SUBINDO = a IA ficando mais forte)
os.makedirs('checkpoints', exist_ok=True)

class Amigavel(BaseCallback):
    def __init__(self, log_every, ckpt_every):
        super().__init__(); self.log_every = log_every; self.ckpt_every = ckpt_every; self.n = 0
    def _on_step(self):
        if self.num_timesteps - getattr(self, '_last', 0) >= self.log_every:
            self._last = self.num_timesteps
            buf = self.model.ep_info_buffer
            r = float(np.mean([e['r'] for e in buf])) if buf else 0.0
            print(f'   passos {self.num_timesteps:>10,}   lucro medio/mao = {r:+7.1f} fichas')
        if self.num_timesteps // self.ckpt_every > self.n:
            self.n += 1
            p = f'checkpoints/poker_ppo_{self.num_timesteps}'
            self.model.save(p)
            print(f'        >>> checkpoint salvo: {p}.zip  (= um nivel de dificuldade)')
        return True

box('TREINANDO...', [
    'lucro medio/mao > 0 = a IA vence versoes anteriores de si mesma,',
    'ou seja, esta ficando mais forte. (self-play e mais lento que CFR)',
])
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=Amigavel(50_000, CKPT_EVERY))
model.save('checkpoints/poker_ppo_final')
box('TREINO CONCLUIDO', ['Modelo final: checkpoints/poker_ppo_final.zip'])

In [ ]:
# 5) Exporta a politica para ONNX (o backend usa onnxruntime, leve) + publica no HF
class OnnxPolicy(torch.nn.Module):
    def __init__(self, policy):
        super().__init__()
        self.features_extractor = policy.features_extractor
        self.mlp_extractor = policy.mlp_extractor
        self.action_net = policy.action_net
    def forward(self, obs):
        f = self.features_extractor(obs)
        latent_pi, _ = self.mlp_extractor(f)
        return self.action_net(latent_pi)  # logits das 5 acoes

policy = model.policy.to('cpu').eval()
torch.onnx.export(OnnxPolicy(policy), torch.zeros(1, FEATURE_SIZE), 'poker_expert.onnx',
                  input_names=['obs'], output_names=['logits'],
                  dynamic_axes={'obs': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17)
print('ONNX exportado: poker_expert.onnx')

from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
repo = 'felipesp1983/poker-arena-expert'
api.create_repo(repo, repo_type='model', exist_ok=True, private=True)
api.upload_file(path_or_fileobj='poker_expert.onnx', path_in_repo='poker_expert.onnx', repo_id=repo, repo_type='model')
api.upload_file(path_or_fileobj='checkpoints/poker_ppo_final.zip', path_in_repo='poker_ppo_final.zip', repo_id=repo, repo_type='model')
box('PUBLICADO NO HF HUB', [f'modelo: {repo}', 'O backend (MLBot) vai baixar o poker_expert.onnx daqui.'])

## Próximo passo (Fase 4)
Com o `poker_expert.onnx` no HF Hub, eu implemento o **`MLBot`** no backend: ele baixa o ONNX, roda inferência sobre a `Observation` (com a máscara legal) e joga como o nível **Expert** (e os checkpoints intermediários viram **Master**). Aí avaliamos a força por **cross-play** (bb/100) — com base científica de verdade.